In [ ]:
# Config
NUM_CHANNELS = 1
SAMPLE_RATE = 44100
CHUNK_SIZE = 2048
FILE_CLEANUP_RATE = 0 # (in seconds), use 0 to disable automatic cleanup of the files.
NOISE_THRESHOLD = 0.01 
MAIN_PATH = "Records/"
MODEL_PATH = "Versions/CLSTM_v2"
BOARD_NAME = "Esp32Mood"
COM_PORT = "COM5"
labels = ["ANG", "DIS", "FEA", "HAP", "NEU", "SAD", "CAL", "SUR"]
color_map = {
  "ANG": [255, 0, 0],
  "DIS": [255, 78, 3],
  "FEA": [128, 0, 190],
  "HAP": [242, 187, 5],
  "NEU": [245, 245, 245],
  "SAD": [2, 0, 196],
  "CAL": [162, 162, 252],
  "SUR": [50, 255, 140]
}


import ipywidgets as widgets
from  IPython.display import display 
from threading import Thread
from queue import Queue as Q
import os
import pyaudio as pa
import wave
import time
import numpy as np
from NeuralNets.audio_data import extract_mel
from NeuralNets.utils import remove_gaps
import matplotlib.pyplot as plt
import serial as ser


# Importing Model and completing setup
from NeuralNets.model import Model
emotion_model = Model.load(MODEL_PATH)
std = np.load("Versions/mel_std.npy")
mean = np.load("Versions/mel_mean.npy")

serialBT = ser.Serial()
serialBT.port = COM_PORT
serialBT.baudrate = 115200
serialBT.timeout = 1
serialBT.write_timeout = 2

transmit = Q()
recordings = Q()
file_to_emotion_q = Q()
file_list = []
stop_cleanup = False

Output_Signal = widgets.Output()
Output_Emotion = widgets.HTML()



# Setting up PyAudio stream with the callback
def callback(in_data, fram_count, time_info, status):
  recordings.put(in_data)
  return (None, pa.paContinue)

p = pa.PyAudio()
stream = p.open(format=p.get_format_from_width(width=2),
  channels=NUM_CHANNELS,
  input=True,
  rate=SAMPLE_RATE,
  frames_per_buffer=CHUNK_SIZE,
  input_device_index=1,
  stream_callback=callback, 
  start=False)

def get_emotion_from_file(file_to_emotion_q):
  while True:
    filename = file_to_emotion_q.get()
    if filename is None: break
    time.sleep(0.1)        
    if os.path.exists(filename):
      mel = extract_mel(filename, None, target_frames=240, filter_N=40, NFFT=2048, augment=False)
      mel = (mel - mean) / (std + 1e-6)


      
      prediction = emotion_model.inference(mel)
      probs = prediction[0] if len(prediction.shape) > 1 else prediction

      # ESP expects a string with the X terminator with the probabilities separated by commas.
      payload_str = f"{probs[0]:.4f},{probs[1]:.4f},{probs[2]:.4f},{probs[3]:.4f},{probs[4]:.4f},{probs[5]:.4f},{probs[6]:.4f},{probs[7]:.4f}X"

      #print(f"Sending to ESP32: {payload_str}")
      if serialBT.is_open:
        try:
          serialBT.write(payload_str.encode('utf-8'))
        except Exception as e:
          print(f"failed to send data over Bluetooth: {e}")


      

def cleanup_files(file_list):
  while not stop_cleanup:
    time.sleep(FILE_CLEANUP_RATE)
    if len(file_list) > 0:
      file_to_remove = file_list.pop(0)
      if os.path.exists(file_to_remove):
        os.remove(file_to_remove)
  print("File cleanup thread exited.")

def process_recording(recordings_queue):

  bytes_to_record = SAMPLE_RATE * NUM_CHANNELS * 2 * 4
  buffer = bytearray()

  while not transmit.empty():

    try:
      data = recordings_queue.get(timeout=0.01)
      buffer.extend(data)
    except:
      pass


    if len(buffer) >= bytes_to_record:
      split = bytes_to_record
      if split %2 != 0:
        split -= 1


      signal_segment = buffer[:split]
      buffer = buffer[split:]

      audio_data = np.frombuffer(signal_segment, dtype=np.int16).astype(np.float32)
      audio_data = audio_data[::2]

      fade_samples = int(SAMPLE_RATE * 0.05)
      if len(audio_data) > fade_samples:
        fade_f = np.linspace(1.0, 0.0, fade_samples)
        audio_data[-fade_samples:] *= fade_f

      audio_data /= 32768.0 
      audio_data = remove_gaps(audio_data, threshold=NOISE_THRESHOLD, window_duration=0.5)

      if audio_data.size == 0:
        continue
      audio_data = np.clip(audio_data, -1.0, 1.0)
      audio_data = (audio_data * 32767).astype(np.int16).tobytes()




      filename = f"{MAIN_PATH}recording_{int(time.gmtime().tm_mday)}_{int(time.gmtime().tm_hour)}_{int(time.gmtime().tm_min)}_{int(time.gmtime().tm_sec)}.wav"
      

      with wave.open(filename, 'wb') as wf:
        wf.setnchannels(NUM_CHANNELS)
        wf.setsampwidth(p.get_sample_size(pa.paInt16))
        wf.setframerate(SAMPLE_RATE // 2)
        wf.writeframes(audio_data)

      file_to_emotion_q.put(filename)
      if (FILE_CLEANUP_RATE > 0): file_list.append(filename)




def start_recording(signal):
  '''if not serialBT.is_open:
    try:
      serialBT.open()
      print("Bluetooth connected!")
    except Exception as e:
      print(f"Failed to open Bluetooth: {e}")'''


  transmit.put(True)

  thread_save_audio = Thread(target=process_recording, args=(recordings,))
  thread_map_emotions = Thread(target=get_emotion_from_file, args=(file_to_emotion_q,))


  thread_save_audio.start()
  thread_map_emotions.start()
  stream.start_stream()

  if FILE_CLEANUP_RATE > 0:
    stop_cleanup = False
    cleanup_thread = Thread(target=cleanup_files, args=(file_list,))
    cleanup_thread.start()
  with Output_Signal:
    display("Recording began")


def stop_recording(signal):
  transmit.get()
  stream.stop_stream()
  file_to_emotion_q.put(None)

  if serialBT.is_open:
    serialBT.close()
    print("Bluetooth disconnected safely.")

  with Output_Signal:
    display("Recording ended")

def clear(data):
  stop_cleanup = True
  Output_Signal.clear_output()


button_start_recording = widgets.Button(
  description = "Begin recording",
  disabled = False,
  button_style="success",
  icon="microphone"
)

button_stop_recording = widgets.Button(
  description = "Stop recording",
  disabled = False,
  button_style= "danger",
  icon = "stop"
)

button_clear_Output = widgets.Button(
  description = "Clear output",
  disabled = False,
  button_style = "info",
  icon = "minus"
)



button_clear_Output.on_click(clear)
button_start_recording.on_click(start_recording)
button_stop_recording.on_click(stop_recording)
display(button_start_recording, button_stop_recording, button_clear_Output, Output_Signal)


Button(button_style='success', description='Begin recording', icon='microphone', style=ButtonStyle())

Button(button_style='danger', description='Stop recording', icon='stop', style=ButtonStyle())

Button(button_style='info', description='Clear output', icon='minus', style=ButtonStyle())

Output()

In [ ]:
# Run this cell to list all audio input devices and serial ports.

import pyaudio as pa
import serial as ser
from serial.tools import list_ports
p = pa.PyAudio()
for i in range(p.get_device_count()):
  print(p.get_device_info_by_index(i))

p.terminate()

comports = ser.tools.list_ports.comports()
for item in comports:
  print( item.name )
  print( item.description )
  print( item.hwid )
  print( item.vid )
  print( item.pid )
  print( item.serial_number )
  print( item.location )
  print( item.manufacturer )
  print( item.product )
  print( item.interface )
